# MiniLM Semantic Model



In [22]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [23]:
df = pd.read_csv("../data/processed/customer_support_tickets_en.csv")

df = df.dropna(subset=["text", "queue"]).copy()

df = df[
    (df["text"].str.strip() != "") &
    (df["queue"].str.strip() != "")
].reset_index(drop=True)

print(df.shape)

(16338, 7)


In [24]:
#split
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["queue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing :", len(X_test))

Training: 13070
Testing : 3268


In [25]:
#load minilm
model = SentenceTransformer("all-MiniLM-L6-v2")
print("MiniLM loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM loaded successfully.


In [27]:
#training embeddings
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True,
    batch_size=32
)

X_test_embeddings = model.encode(
    X_test.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Training embeddings:", X_train_embeddings.shape)
print("Testing embeddings :", X_test_embeddings.shape)

Batches:   0%|          | 0/409 [00:00<?, ?it/s]

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Training embeddings: (13070, 384)
Testing embeddings : (3268, 384)


In [28]:
#train classifier
minilm_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

minilm_classifier.fit(
    X_train_embeddings,
    y_train
)

print("MiniLM + Logistic Regression training completed.")

MiniLM + Logistic Regression training completed.


In [29]:
#predictions
minilm_pred = minilm_classifier.predict(
    X_test_embeddings
)

In [30]:
minilm_accuracy = accuracy_score(
    y_test,
    minilm_pred
)

minilm_precision = precision_score(
    y_test,
    minilm_pred,
    average="weighted",
    zero_division=0
)

minilm_recall = recall_score(
    y_test,
    minilm_pred,
    average="weighted",
    zero_division=0
)

minilm_macro_f1 = f1_score(
    y_test,
    minilm_pred,
    average="macro",
    zero_division=0
)

minilm_weighted_f1 = f1_score(
    y_test,
    minilm_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy      : {minilm_accuracy:.4f}")
print(f"Precision     : {minilm_precision:.4f}")
print(f"Recall        : {minilm_recall:.4f}")
print(f"Macro F1      : {minilm_macro_f1:.4f}")
print(f"Weighted F1   : {minilm_weighted_f1:.4f}")

Accuracy      : 0.3023
Precision     : 0.4000
Recall        : 0.3023
Macro F1      : 0.2913
Weighted F1   : 0.3165


In [ ]:
print(
    classification_report(
        y_test,
        minilm_pred,
        zero_division=0
    )
)

                                 precision    recall  f1-score   support

           Billing and Payments       0.70      0.65      0.67       319
               Customer Service       0.31      0.19      0.24       482
                General Inquiry       0.06      0.45      0.11        47
                Human Resources       0.10      0.49      0.17        70
                     IT Support       0.24      0.25      0.25       388
                Product Support       0.39      0.15      0.22       615
          Returns and Exchanges       0.18      0.30      0.22       164
            Sales and Pre-Sales       0.14      0.48      0.21       103
Service Outages and Maintenance       0.35      0.67      0.46       133
              Technical Support       0.53      0.27      0.36       947

                       accuracy                           0.30      3268
                      macro avg       0.30      0.39      0.29      3268
                   weighted avg       0.40      0

In [31]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression + TF-IDF",
        "Accuracy": 0.5294,
        "Macro F1": 0.5306
    },
    {
        "Model": "SVM + Word TF-IDF (1,3)",
        "Accuracy": 0.738372,
        "Macro F1": 0.751939
    },
    {
        "Model": "MiniLM + Logistic Regression",
        "Accuracy": minilm_accuracy,
        "Macro F1": minilm_macro_f1
    }
])

comparison

,Model,Accuracy,Macro F1
0,Logistic Regression + TF-IDF,0.529400,0.530600
1,"SVM + Word TF-IDF (1,3)",0.738372,0.751939
2,MiniLM + Logistic Regression,0.302326,0.291283


In [32]:
comparison.to_csv(
    "../reports/final_model_comparison.csv",
    index=False
)

print("Model comparison saved.")

Model comparison saved.


In [33]:
#saving minilm artifacts
joblib.dump(
    minilm_classifier,
    "../models/minilm_logistic_classifier.pkl"
)
np.save(
    "../models/minilm_train_embeddings.npy",
    X_train_embeddings
)

np.save(
    "../models/minilm_test_embeddings.npy",
    X_test_embeddings
)
